# Phase 4 — Held-out evaluation and error analysis

This notebook evaluates the best saved LayoutLMv3 checkpoint on SROIE's untouched official test split. It reports token/entity metrics on the dataset's reference OCR words, then runs the model on the cached Tesseract words from Phase 2 so its field-level results can be compared with the rule-based baseline on identical OCR input.

The distinction matters: the reference-word pass measures extraction with the dataset's supplied words and boxes, while the shared-OCR pass measures the practical OCR-plus-extraction path. Neither pass trains or selects a model. LayoutLMv3 token-classification APIs follow the official [Transformers 5.19 LayoutLMv3 docs](https://huggingface.co/docs/transformers/v5.19.0/model_doc/layoutlmv3).

## Before running
Complete Phase 3 and confirm that Drive contains `models/layoutlmv3-sroie-best/`, `checkpoints/layoutlmv3-sroie/training_history.json`, and the Phase 1 test data. Complete Phase 2 to compare against `outputs/ocr_test_predictions.jsonl`.

Run this notebook in Colab with the same Drive account. It does inference only; a GPU speeds up evaluation, but CPU inference is also supported. The test set remains held out.

## Cell 1 — Install compatible libraries and inspect runtime

In [ ]:
%pip -q install --upgrade "transformers==5.19.0" "huggingface-hub>=1.31,<2.0" safetensors pandas matplotlib tqdm

import sys, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("No GPU detected; evaluation will run on CPU.")

## Cell 2 — Mount Drive and check required Phase 1–3 files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, shutil, time, math
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch

DRIVE_ROOT = Path("/content/drive/MyDrive/intelligent-document-ai")
DATA_PROCESSED = DRIVE_ROOT / "data" / "processed"
ANNOTATIONS = DRIVE_ROOT / "data" / "annotations"
MODEL_DIR = DRIVE_ROOT / "models" / "layoutlmv3-sroie-best"
HISTORY_PATH = DRIVE_ROOT / "checkpoints" / "layoutlmv3-sroie" / "training_history.json"
OCR_CACHE_PATH = DRIVE_ROOT / "outputs" / "ocr_test_predictions.jsonl"
METRICS_DIR = DRIVE_ROOT / "metrics"
OUTPUTS_DIR = DRIVE_ROOT / "outputs"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

required = [
    DATA_PROCESSED / "test.jsonl",
    ANNOTATIONS / "phase1_metadata.json",
    MODEL_DIR / "config.json",
    OCR_CACHE_PATH,
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing Phase 4 inputs. Complete the named earlier phases first:\n" + "\n".join(missing)
    )
print("Best checkpoint:", MODEL_DIR)
print("Phase 2 OCR cache:", OCR_CACHE_PATH)

## Cell 3 — Load the official test records and identify exact duplicates
Only the official test split is eligible for evaluation. Prepared filenames can collide because row IDs restart in each split, so this cell compares image bytes by SHA-256. Any test image whose bytes also occur in train or validation is excluded from scoring and written to an audit CSV; the original `test.jsonl` remains unchanged.


In [ ]:
def read_jsonl(path):
    rows = []
    with Path(path).open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Bad JSON at {path}:{line_number}") from exc
    return rows

phase1_metadata = json.loads((ANNOTATIONS / "phase1_metadata.json").read_text(encoding="utf-8"))
LABELS = phase1_metadata["label_names"]
LABEL2ID = {label: index for index, label in enumerate(LABELS)}
ID2LABEL = {index: label for label, index in LABEL2ID.items()}
all_test_records = read_jsonl(DATA_PROCESSED / "test.jsonl")
if not all_test_records:
    raise ValueError("The prepared test split is empty.")
all_test_ids = {str(row["document_id"]) for row in all_test_records}

import hashlib
def image_sha256(record):
    image_path = Path(record["image_path"])
    if not image_path.is_file():
        raise FileNotFoundError(f"Missing image for duplicate check: {image_path}")
    return hashlib.sha256(image_path.read_bytes()).hexdigest()

# Keep every earlier-split match so the exclusions can be reviewed.
earlier_by_hash = {}
for split_name in ("train", "validation"):
    split_path = DATA_PROCESSED / f"{split_name}.jsonl"
    if split_path.exists():
        for record in read_jsonl(split_path):
            digest = image_sha256(record)
            earlier_by_hash.setdefault(digest, []).append((split_name, record))

excluded_test_hashes = set()
duplicate_audit_rows = []
for test_row in all_test_records:
    digest = image_sha256(test_row)
    for split_name, earlier_row in earlier_by_hash.get(digest, []):
        excluded_test_hashes.add(digest)
        duplicate_audit_rows.append({
            "test_document_id": str(test_row["document_id"]),
            "test_image_path": str(test_row["image_path"]),
            "matching_split": split_name,
            "matching_document_id": str(earlier_row["document_id"]),
            "matching_image_path": str(earlier_row["image_path"]),
            "sha256": digest,
        })

test_records = [row for row in all_test_records if image_sha256(row) not in excluded_test_hashes]
test_ids = {str(row["document_id"]) for row in test_records}
if not test_records:
    raise ValueError("Every official test image matches train/validation; no leakage-free test records remain to evaluate.")

duplicate_audit_path = OUTPUTS_DIR / "test_duplicate_exclusions.csv"
pd.DataFrame(duplicate_audit_rows, columns=[
    "test_document_id", "test_image_path", "matching_split",
    "matching_document_id", "matching_image_path", "sha256",
]).to_csv(duplicate_audit_path, index=False)
print(f"Official test documents: {len(all_test_records)}")
print(f"Excluded test documents with exact train/validation image matches: {len(all_test_records) - len(test_records)}")
print(f"Leakage-free documents evaluated: {len(test_records)}")
if duplicate_audit_rows:
    display(pd.DataFrame(duplicate_audit_rows).drop_duplicates("test_document_id").head(30))
    print("Full duplicate audit:", duplicate_audit_path)
print("Labels:", LABELS)


## Cell 4 — Load the saved processor and best model

In [ ]:
from transformers import AutoProcessor, LayoutLMv3ForTokenClassification

processor = AutoProcessor.from_pretrained(MODEL_DIR, apply_ocr=False)
model = LayoutLMv3ForTokenClassification.from_pretrained(MODEL_DIR)
model.eval()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE)
MAX_LENGTH = min(512, int(model.config.max_position_embeddings))
print("Model:", type(model).__name__)
print("Device:", DEVICE, "| max sequence length:", MAX_LENGTH)
if model.config.num_labels != len(LABELS):
    raise ValueError(f"Checkpoint has {model.config.num_labels} labels, Phase 1 has {len(LABELS)}")
print("Checkpoint label map:", model.config.id2label)

## Cell 5 — Define BIO, entity, and field metrics
Entity F1 uses exact BIO spans (document, start, end, field type). Field exact match normalizes case and punctuation consistently with Phase 2.

In [ ]:
FIELDS = ("COMPANY", "DATE", "ADDRESS", "TOTAL")

def close_bio_span(spans, document_index, active_label, active_start, end_index):
    if active_label is not None:
        spans.add((document_index, active_start, end_index, active_label))

def bio_spans(tags, document_index):
    spans = set()
    active_label, active_start = None, None
    for position, tag in enumerate(tags):
        if tag == "O":
            close_bio_span(spans, document_index, active_label, active_start, position)
            active_label, active_start = None, None
        elif tag.startswith("B-"):
            close_bio_span(spans, document_index, active_label, active_start, position)
            active_label, active_start = tag[2:], position
        elif tag.startswith("I-") and active_label == tag[2:]:
            continue
        elif tag.startswith("I-"):
            # Treat an orphan/mismatched I-tag as a new entity start.
            close_bio_span(spans, document_index, active_label, active_start, position)
            active_label, active_start = tag[2:], position
        else:
            close_bio_span(spans, document_index, active_label, active_start, position)
            active_label, active_start = None, None
    close_bio_span(spans, document_index, active_label, active_start, len(tags))
    return spans

def entity_metrics(predictions, references):
    pred_set, gold_set = set(), set()
    for index, (pred, gold) in enumerate(zip(predictions, references)):
        pred_set.update(bio_spans(pred, index))
        gold_set.update(bio_spans(gold, index))
    tp = len(pred_set & gold_set)
    fp, fn = len(pred_set - gold_set), len(gold_set - pred_set)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "entity_f1": f1,
            "true_positive_entities": tp, "false_positive_entities": fp,
            "false_negative_entities": fn}

def fields_from_bio(words, tags):
    if len(words) != len(tags):
        raise ValueError("Words and BIO tags must have equal lengths.")
    spans = {field: [] for field in FIELDS}
    active_field, active_words = None, []
    def finish():
        if active_field in spans and active_words:
            spans[active_field].append(" ".join(active_words))
    for word, tag in zip(words, tags):
        if tag == "O":
            finish()
            active_field, active_words = None, []
        elif tag.startswith("B-"):
            finish()
            active_field, active_words = tag[2:], [word]
        elif tag.startswith("I-") and active_field == tag[2:]:
            active_words.append(word)
        elif tag.startswith("I-"):
            finish()
            active_field, active_words = tag[2:], [word]
        else:
            finish()
            active_field, active_words = None, []
    finish()
    return {field: " ".join(spans[field]).strip() for field in FIELDS}

def normalize_value(value):
    import re
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())

def field_metrics(rows, prediction_key="prediction", gold_key="gold"):
    counts = {field: {"correct": 0, "eligible": 0, "tp": 0, "fp": 0, "fn": 0}
              for field in FIELDS}
    document_exact = 0
    for row in rows:
        doc_correct = True
        for field in FIELDS:
            pred = normalize_value(row[prediction_key].get(field, ""))
            gold = normalize_value(row[gold_key].get(field, ""))
            c = counts[field]
            if gold:
                c["eligible"] += 1
                if pred == gold:
                    c["correct"] += 1
                    c["tp"] += 1
                else:
                    doc_correct = False
                    c["fn"] += 1
                    if pred:
                        c["fp"] += 1
            elif pred:
                doc_correct = False
                c["fp"] += 1
        document_exact += int(doc_correct)

    rows_out = []
    all_correct = all_eligible = total_tp = total_fp = total_fn = 0
    for field, c in counts.items():
        p = c["tp"] / (c["tp"] + c["fp"]) if c["tp"] + c["fp"] else 0.0
        r = c["tp"] / (c["tp"] + c["fn"]) if c["tp"] + c["fn"] else 0.0
        f1 = 2 * p * r / (p + r) if p + r else 0.0
        exact = c["correct"] / c["eligible"] if c["eligible"] else 0.0
        rows_out.append({"field": field, "exact_match_accuracy": exact,
                         "precision": p, "recall": r, "f1": f1,
                         "correct": c["correct"], "gold_present": c["eligible"]})
        all_correct += c["correct"]
        all_eligible += c["eligible"]
        total_tp += c["tp"]
        total_fp += c["fp"]
        total_fn += c["fn"]
    p = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0.0
    r = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0.0
    micro_f1 = 2 * p * r / (p + r) if p + r else 0.0
    summary = {
        "field_exact_match_accuracy": all_correct / all_eligible if all_eligible else 0.0,
        "document_all_fields_exact_match": document_exact / len(rows) if rows else 0.0,
        "field_value_precision": p, "field_value_recall": r, "field_value_f1": micro_f1,
    }
    return summary, rows_out

## Cell 6 — Evaluate on SROIE reference words and boxes
This pass measures token extraction given SROIE's supplied OCR transcript and boxes. For each document, the processor aligns BIO labels to first subwords; special and continuation positions are ignored. Any sequence truncation is counted.

In [ ]:
MAX_LENGTH = min(512, int(model.config.max_position_embeddings))
reference_predictions, reference_gold = [], []
reference_field_rows = []
truncated_reference_docs = 0
LOCAL_TEST_DIR = Path("/content/idai_phase4_test_images")
LOCAL_TEST_DIR.mkdir(parents=True, exist_ok=True)

def local_test_image(row):
    source = Path(row["image_path"])
    target = LOCAL_TEST_DIR / source.name
    if not target.exists():
        shutil.copy2(source, target)
    return target

model.eval()
for row in tqdm(test_records, desc="Reference-word evaluation"):
    image_path = local_test_image(row)
    image = Image.open(image_path).convert("RGB")
    true_ids = [LABEL2ID[tag] for tag in row["ner_tags"]]
    encoded = processor(
        images=image, text=row["words"], boxes=row["bboxes"], word_labels=true_ids,
        truncation=True, max_length=MAX_LENGTH, padding="max_length", return_tensors="pt",
    )
    aligned_labels = encoded["labels"][0]
    word_mask = aligned_labels.ne(-100)
    if int(word_mask.sum()) < len(row["words"]):
        truncated_reference_docs += 1
    inputs = {key: value.to(DEVICE) for key, value in encoded.items() if key != "labels"}
    with torch.inference_mode():
        outputs = model(**inputs)
    predicted_ids = outputs.logits[0][word_mask.to(outputs.logits.device)].argmax(dim=-1).cpu().tolist()
    gold_ids = aligned_labels[word_mask].tolist()
    pred_tags = [ID2LABEL[int(value)] for value in predicted_ids]
    gold_tags = [ID2LABEL[int(value)] for value in gold_ids]
    visible_words = row["words"][:len(gold_tags)]
    reference_predictions.append(pred_tags)
    reference_gold.append(gold_tags)
    reference_field_rows.append({
        "document_id": str(row["document_id"]),
        "prediction": fields_from_bio(visible_words, pred_tags),
        "gold": fields_from_bio(visible_words, gold_tags),
        "words_evaluated": len(gold_tags),
        "words_in_document": len(row["words"]),
    })

reference_entity_summary = entity_metrics(reference_predictions, reference_gold)
reference_field_summary, reference_per_field = field_metrics(reference_field_rows)
print("Reference-word entity metrics:", reference_entity_summary)
print("Reference-word field metrics:", reference_field_summary)
print("Documents with truncated word labels:", truncated_reference_docs)

## Cell 7 — Evaluate on cached Phase 2 Tesseract OCR and compare baseline
Both the learned model and the rule-based baseline receive the same Phase 2 OCR tokens and normalized boxes. This supports a fair field-value comparison on the end-to-end OCR pipeline. Token entity F1 is not computed here because Tesseract token boundaries do not align one-to-one with SROIE's reference annotation words.

In [ ]:
ocr_rows = read_jsonl(OCR_CACHE_PATH)
ocr_by_id = {str(row["document_id"]): row for row in ocr_rows}
if set(ocr_by_id) != all_test_ids:
    missing_ids = sorted(all_test_ids - set(ocr_by_id))
    extra_ids = sorted(set(ocr_by_id) - all_test_ids)
    raise ValueError(
        f"Phase 2 OCR cache must match official test IDs. Missing={missing_ids[:5]}, extra={extra_ids[:5]}"
    )

def predict_on_ocr_tokens(ocr_row, measure_latency=False):
    tokens = ocr_row["ocr_tokens"]
    words = [token["text"] for token in tokens]
    boxes = [token["bbox"] for token in tokens]
    if not words:
        return {field: "" for field in FIELDS}, [], 0.0
    local_image_path = local_test_image({"image_path": ocr_row["image_path"]})
    if measure_latency and DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    image = Image.open(local_image_path).convert("RGB")
    dummy_labels = [LABEL2ID["O"]] * len(words)
    encoded = processor(
        images=image, text=words, boxes=boxes, word_labels=dummy_labels,
        truncation=True, max_length=MAX_LENGTH, padding="max_length", return_tensors="pt",
    )
    label_mask = encoded["labels"][0].ne(-100)
    inputs = {key: value.to(DEVICE) for key, value in encoded.items() if key != "labels"}
    with torch.inference_mode():
        outputs = model(**inputs)
    predicted_ids = outputs.logits[0][label_mask.to(outputs.logits.device)].argmax(dim=-1).cpu().tolist()
    if measure_latency and DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start if measure_latency else 0.0
    pred_tags = [ID2LABEL[int(value)] for value in predicted_ids]
    visible_words = words[:len(pred_tags)]
    return fields_from_bio(visible_words, pred_tags), pred_tags, elapsed

# Warm up the model once; warmup time is excluded from measured latencies.
first_ocr = ocr_by_id[str(test_records[0]["document_id"])]
_ = predict_on_ocr_tokens(first_ocr, measure_latency=False)

ocr_model_rows = []
latencies = []
for test_row in tqdm(test_records, desc="Model on cached Tesseract OCR"):
    doc_id = str(test_row["document_id"])
    ocr_row = ocr_by_id[doc_id]
    prediction, pred_tags, elapsed = predict_on_ocr_tokens(ocr_row, measure_latency=True)
    latencies.append(elapsed)
    gold = fields_from_bio(test_row["words"], test_row["ner_tags"])
    ocr_model_rows.append({
        "document_id": doc_id,
        "prediction": prediction,
        "gold": gold,
        "baseline_prediction": ocr_row["prediction"],
        "predicted_tags": pred_tags,
        "ocr_word_count": len(ocr_row["ocr_tokens"]),
    })

model_ocr_summary, model_ocr_per_field = field_metrics(ocr_model_rows)
baseline_rows = [
    {"document_id": row["document_id"], "prediction": row["baseline_prediction"], "gold": row["gold"]}
    for row in ocr_model_rows
]
baseline_summary, baseline_per_field = field_metrics(baseline_rows)
comparison = pd.DataFrame([
    {"system": "Rule baseline", **baseline_summary},
    {"system": "LayoutLMv3 on same Tesseract OCR", **model_ocr_summary},
])
display(comparison)
latency_summary = {
    "documents": len(latencies),
    "mean_seconds_per_document": float(np.mean(latencies)) if latencies else 0.0,
    "median_seconds_per_document": float(np.median(latencies)) if latencies else 0.0,
    "p95_seconds_per_document": float(np.percentile(latencies, 95)) if latencies else 0.0,
    "documents_per_second": float(1 / np.mean(latencies)) if latencies and np.mean(latencies) > 0 else 0.0,
    "timed_scope": "image loading, processor encoding, and model forward pass; cached Tesseract OCR is excluded",
}
print("Inference latency:", latency_summary)

## Cell 8 — Plot training and validation curves
The history is loaded from the Phase 3 checkpoint folder. If it is missing, this cell reports that fact rather than inventing values.

In [ ]:
import matplotlib.pyplot as plt
if HISTORY_PATH.is_file():
    history = json.loads(HISTORY_PATH.read_text(encoding="utf-8"))
    history_df = pd.DataFrame(history)
    display(history_df)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train loss")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="validation loss")
    axes[0].set_title("Training and validation loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].plot(history_df["epoch"], history_df["entity_f1"], marker="o")
    axes[1].set_title("Validation entity-level F1")
    axes[1].set_xlabel("Epoch")
    plt.tight_layout()
    fig.savefig(OUTPUTS_DIR / "training_validation_curves.png", dpi=150)
    plt.show()
else:
    print("No Phase 3 training history found at:", HISTORY_PATH)

## Cell 9 — Inspect incorrect fields and token spans
The field error table focuses on the shared-OCR model versus its reference values. Token-span errors below compare reference-word test tags to model tags and are a small sample for inspection.

In [ ]:
field_error_rows = []
for row in ocr_model_rows:
    for field in FIELDS:
        pred = row["prediction"].get(field, "")
        gold = row["gold"].get(field, "")
        if normalize_value(pred) != normalize_value(gold):
            field_error_rows.append({
                "document_id": row["document_id"], "field": field,
                "prediction": pred, "reference": gold,
            })
field_errors_df = pd.DataFrame(field_error_rows)
print("Shared-OCR field errors:", len(field_error_rows))
display(field_errors_df.head(30))

token_error_rows = []
for row, pred_tags, gold_tags in zip(test_records, reference_predictions, reference_gold):
    visible = min(len(row["words"]), len(pred_tags), len(gold_tags))
    for index in range(visible):
        if pred_tags[index] != gold_tags[index]:
            token_error_rows.append({
                "document_id": str(row["document_id"]),
                "word": row["words"][index],
                "gold_tag": gold_tags[index],
                "predicted_tag": pred_tags[index],
            })
token_errors_df = pd.DataFrame(token_error_rows)
print("Reference-word tag errors:", len(token_error_rows))
display(token_errors_df.head(40))

## Cell 10 — Save evaluation outputs to Drive

In [ ]:
report = {
    "dataset": "darentang/sroie",
    "evaluated_split": "official test",
    "documents": len(test_records),
    "official_test_documents_before_duplicate_exclusion": len(all_test_records),
    "exact_train_validation_duplicates_excluded": len(all_test_records) - len(test_records),
    "duplicate_exclusion_audit": str(duplicate_audit_path),
    "checkpoint": str(MODEL_DIR),
    "metrics": {
        "reference_words_and_boxes": {
            "entity_metrics": reference_entity_summary,
            "field_summary": reference_field_summary,
            "per_field": reference_per_field,
            "documents_with_truncated_labels": truncated_reference_docs,
        },
        "shared_phase2_tesseract_ocr": {
            "layoutlmv3_field_summary": model_ocr_summary,
            "layoutlmv3_per_field": model_ocr_per_field,
            "rule_baseline_field_summary": baseline_summary,
            "rule_baseline_per_field": baseline_per_field,
        },
        "inference_latency": latency_summary,
    },
    "assumptions": [
        "Reference-word entity metrics use the dataset-supplied words, boxes, and BIO labels.",
        "The field comparison feeds the same cached Phase 2 Tesseract OCR tokens to both systems.",
        "Field equality is case-insensitive after removing non-alphanumeric characters.",
        "Latency includes image loading, processor encoding, and the model forward pass; Tesseract OCR is excluded.",
    ],
}
(METRICS_DIR / "layoutlmv3_test_evaluation.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
comparison.to_csv(OUTPUTS_DIR / "baseline_vs_layoutlmv3_test.csv", index=False)
field_errors_df.to_csv(OUTPUTS_DIR / "layoutlmv3_field_errors.csv", index=False)
token_errors_df.to_csv(OUTPUTS_DIR / "layoutlmv3_token_errors.csv", index=False)

prediction_export = []
for row in ocr_model_rows:
    prediction_export.append({
        "document_id": row["document_id"],
        "prediction": row["prediction"],
        "gold": row["gold"],
        "baseline_prediction": row["baseline_prediction"],
        "ocr_word_count": row["ocr_word_count"],
    })
(OUTPUTS_DIR / "layoutlmv3_test_predictions.json").write_text(
    json.dumps(prediction_export, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Saved duplicate audit:", duplicate_audit_path)
print("Saved metrics:", METRICS_DIR / "layoutlmv3_test_evaluation.json")
print("Saved comparison:", OUTPUTS_DIR / "baseline_vs_layoutlmv3_test.csv")
print("Saved field errors:", OUTPUTS_DIR / "layoutlmv3_field_errors.csv")
print("Saved token errors:", OUTPUTS_DIR / "layoutlmv3_token_errors.csv")

## Reading the report and limitations
The reference-word entity metrics isolate extraction quality with the dataset's supplied OCR words and boxes. The baseline comparison uses Tesseract OCR for both systems and is a more practical pipeline comparison. Review the assumptions and error files before interpreting results. The evaluated test set is held out; do not use it to tune the model. No accuracy target is guaranteed, and this receipt model does not establish invoice or bank-statement support.